# 🤖 Notebook 02: Segmentation & CLV Model Training
### Project: Professional Customer Segmentation & CLV Prediction

This notebook performs unsupervised clustering (K-Means/DBSCAN) and trains supervised regression models to predict future CLV.

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score

# Base paths
BASE_DIR = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd()
PROCESSED_DATA = os.path.join(BASE_DIR, 'data', 'processed', 'rfm_clv_data.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

df = pd.read_csv(PROCESSED_DATA, index_col=0)
print(f"Loaded RFM data: {df.shape}")

Loaded RFM data: (4933, 4)


## 1. Data Transformation
RFM data is highly skewed. We use `PowerTransformer` (Yeo-Johnson) and `StandardScaler` to normalize the distribution for clustering.

In [2]:
features = ['Recency', 'Frequency', 'Monetary']
X_cl = df[features]

# Transform and Scale
pt = PowerTransformer(method='yeo-johnson')
scaler = StandardScaler()

X_transformed = pt.fit_transform(X_cl)
X_scaled = scaler.fit_transform(X_transformed)

with open(os.path.join(MODELS_DIR, 'transformer.pkl'), 'wb') as f:
    pickle.dump(pt, f)
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print("Preprocessing pipelines saved.")

Preprocessing pipelines saved.


## 2. Unsupervised Clustering
### K-Means (k=4)
Dividing customers into: Champions, Loyal, At-Risk, and Lost.

In [3]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, df['Cluster'])
print(f"K-Means Silhouette Score: {sil_score:.4f}")

with open(os.path.join(MODELS_DIR, 'kmeans_model.pkl'), 'wb') as f:
    pickle.dump(kmeans, f)

K-Means Silhouette Score: 0.3804


### DBSCAN (Outlier/VIP Detection)
Isolating genuine outliers that K-Means might force into a cluster.

In [4]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
df['DBSCAN_Outlier'] = dbscan.fit_predict(X_scaled)

vip_count = len(df[df['DBSCAN_Outlier'] == -1])
print(f"DBSCAN detected {vip_count} potential VIP outliers.")

with open(os.path.join(MODELS_DIR, 'dbscan_model.pkl'), 'wb') as f:
    pickle.dump(dbscan, f)

DBSCAN detected 37 potential VIP outliers.


## 3. Supervised CLV Regression
We predict `CLV_Target` using the transformed RFM features + Cluster membership.

In [5]:
# Filter users who had at least some activity (Monetary > 0 is already true from Notebook 01)
X_reg = np.column_stack([X_scaled, df['Cluster']])
y_reg = df['CLV_Target']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Model 1: Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Model 3: Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

models = {'ridge': ridge, 'rf': rf, 'gb': gb}
for name, model in models.items():
    with open(os.path.join(MODELS_DIR, f'{name}_clv_model.pkl'), 'wb') as f:
        pickle.dump(model, f)
    
    y_pred = model.predict(X_test)
    print(f"{name.upper()} - MAE: {mean_absolute_error(y_test, y_pred):.2f}, R2: {r2_score(y_test, y_pred):.4f}")

df.to_csv(PROCESSED_DATA)
print("Training complete for all 3 models.")

RIDGE - MAE: 1354.17, R2: 0.0959
RF - MAE: 708.45, R2: 0.7954
GB - MAE: 667.41, R2: 0.8139
Training complete for all 3 models.
